In [74]:
import pandas as pd

current = pd.read_csv("../data/raw/adas_current.csv", low_memory=False)
archive = pd.read_csv("../data/raw/adas_archive.csv", low_memory=False, encoding="latin-1")

# print(current.head())

print("current:", current.shape)
print("archive:", archive.shape)

current: (2313, 116)
archive: (4027, 137)


In [75]:
cur_cols, arc_cols = set(current.columns), set(archive.columns)
print("Only in current: ", len(sorted(cur_cols - arc_cols)))
print("Only in archive: ", len(sorted(arc_cols-cur_cols)))
print("Shared between the two: ", len(cur_cols & arc_cols))

print("Only in current:", sorted(cur_cols - arc_cols))
print("Only in archive:", sorted(arc_cols - cur_cols))

Only in current:  27
Only in archive:  48
Shared between the two:  89
Only in current: ['Any Air Bags Deployed?', 'Automation Feature Version', 'Automation Feature Version CBI', 'Engagement Status', 'Make - Unknown', 'Roadway-Degraded Surface', 'Roadway-Missing/Degraded Marking', 'Roadway-No Unusual Conditions', 'Roadway-Other-See Narrative', 'Roadway-Traffic Incident', 'Roadway-Unknown', 'Roadway-Wet Surface Condition', 'Roadway-Work Zone', 'Source - Internal Process Review', 'Source - NHTSA VOQ', 'Source - Other - See Narrative', 'Source - Other Entity', 'Source - State or Other Agency', 'VIN Decoded', 'Was Any Vehicle Towed?', 'Weather - Dust Storm', 'Weather - Fog/Smoke/Haze', 'Weather - Partly Cloudy', 'Weather - Severe Hurricane', 'Weather - Structure-Indoor', 'Weather - Unk - See Narrative', 'Were All Passengers Belted?']
Only in archive: ['ADAS/ADS Hardware Version', 'ADAS/ADS Hardware Version - Unk', 'ADAS/ADS Hardware Version CBI', 'ADAS/ADS Software Version', 'ADAS/ADS Softw

In [76]:
current.head(3).T

,0,1,2
Report ID,753-15460,669-16212,1243-16267
Report Version,2,1,1
Reporting Entity,"Kia America, Inc.",Hyundai Motor America,"Subaru of America, Inc."
Report Type,Update,5-Day,5-Day
Report Month,NaN,NaN,NaN
...,...,...,...
Investigating Agency - Unknown,Y,,Y
Within ODD?,"Unknown, see Narrative","Unknown, see Narrative","[REDACTED, MAY CONTAIN CONFIDENTIAL BUSINESS I..."
Within ODD? - CBI,,,Y
Narrative,"**6/13/2026 5-Day Report**\nOn June [XXX], 202...",A Hyundai Motor America associate was operatin...,"[REDACTED, MAY CONTAIN CONFIDENTIAL BUSINESS I..."


In [77]:
# Missing values: which columns are mostly empty?
missing = current.isna().mean().sort_values(ascending=False)
missing.head(20)

Report Month                     0.998271
Report Year                      0.998271
Operating Entity                 0.980977
Investigating Agency             0.957198
Serial Number                    0.218331
CP Contact Area - Rear           0.202767
CP Contact Area - Rear Left      0.202767
CP Contact Area - Rear Right     0.202767
CP Contact Area - Front          0.202767
CP Contact Area - Top            0.202767
CP Contact Area - Front Right    0.202767
CP Contact Area - Front Left     0.202767
CP Contact Area - Left           0.202767
CP Contact Area - Right          0.202767
CP Contact Area - Bottom         0.202767
CP Contact Area - Unknown        0.202767
CP Pre-Crash Movement            0.184609
Address                          0.130999
Zip Code                         0.128405
SV Precrash Speed (MPH)          0.120190
dtype: float64

In [78]:
# Reports vs. crashes 
print(current["Report ID"].nunique(), "unique report IDs out of", len(current), "rows")
print(current["Report Version"].value_counts().head())

2186 unique report IDs out of 2313 rows
Report Version
1    2149
2     141
3      17
4       5
5       1
Name: count, dtype: int64


In [79]:
for col in ["Roadway Type", "Lighting", "Crash With", "Highest Injury Severity Alleged"]:
    print(f"\n--- {col} ---")
    print(current[col].value_counts(dropna=False).head(10))


--- Roadway Type ---
Roadway Type
Highway / Freeway    914
Unknown              833
Street               248
Intersection         234
Rural Road            54
Parking Lot           24
Parking Garage         3
Unpaved Road           2
Traffic Circle         1
Name: count, dtype: int64

--- Lighting ---


KeyError: 'Lighting'

In [ ]:
print([c for c in archive.columns if "light" in c.lower()])

['Lighting']


In [ ]:
for c in sorted(current.columns):
    print(c)

Address
Address - Unknown
Any Air Bags Deployed?
Automation Feature Version
Automation Feature Version CBI
Automation System Engaged?
CP Contact Area - Bottom
CP Contact Area - Front
CP Contact Area - Front Left
CP Contact Area - Front Right
CP Contact Area - Left
CP Contact Area - Rear
CP Contact Area - Rear Left
CP Contact Area - Rear Right
CP Contact Area - Right
CP Contact Area - Top
CP Contact Area - Unknown
CP Pre-Crash Movement
City
City - Unknown
Crash With
Data Availability - Complaints
Data Availability - EDR
Data Availability - No Data
Data Availability - Other
Data Availability - Police Rpt
Data Availability - Telematics
Data Availability - Unknown
Data Availability - Video
Driver / Operator Type
Engagement Status
Highest Injury Severity Alleged
Incident Date
Incident Date - Unknown
Incident Time (24:00)
Incident Time - Unknown
Investigating Agency
Investigating Agency - Unknown
Latitude
Latitude - Unknown
Longitude
Longitude - Unknown
Make
Make - Unknown
Model
Model - Unkn

First findings

- Current file: 2,313 report rows (June 2025 onward), 116 columns. 
- Archive: 4,027 rows (2021 – June 2025), 137 columns. -> 89 columns shared.
- Rows are reports, not crashes: need to keep the latest version per crash.
- Lighting exists only in the archive (removed in the 2025 rule change).
- Crash partner contact areas are ~20% missing, likely crashes without a partner (structural missingness).
- Pre-crash speed is known for ~88% of current reports.
- Missing and confidential values are flagged in separate "- Unknown" and "CBI" columns.